# Mission 01: The Architecture of Argument
**USSE Noosphere, Conversational AI (WS 2026/27)**  |  Role: Core Architect

This notebook runs the squad's Dialogic Engine (`04_dialogic_engine.py`). Prompts and protocol settings are **read from the squad's markdown files** in this folder, so teammates can change them without touching the code:

| File | What the engine reads |
|---|---|
| `01_Alignment_Brief.md` | AURA persona prompt (3.2) and contradiction referee prompt (3.3) |
| `02_Socratic_Protocol_Map.md` | JSON-RPC method per state, error codes, refinement limit and defensibility threshold |
| `03_Galilean_Sandbox.md` | Salviati / Simplicio / Sagredo prompts and the dilemma |

For real LLMs, put `GEMINI_API_KEY` (or `OPENAI_API_KEY` / `ANTHROPIC_API_KEY`) in a `.env` file next to this notebook (never commit it). With no key, the engine runs in offline *mock* mode.

In [1]:
import importlib.util
from pathlib import Path

spec = importlib.util.spec_from_file_location("dialogic_engine", Path("04_dialogic_engine.py"))
engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)

engine.load_dotenv()

PROVIDER = "mock"   # Set to "auto" when you want the notebook to use a real API key, or "mock" for a deterministic offline run.
provider = engine.auto_provider() if PROVIDER == "auto" else PROVIDER

prompts = engine.load_prompts(".")                      # reads the 01 / 02 / 03 markdown files in this folder
logger = engine.RunLogger("logs", prompts.protocol)

print("Provider:", provider)
engine.print_prompt_sources(prompts)

Provider: mock
Prompt sources:
  Protocol (02)            <- 02_Socratic_Protocol_Map.md (methods, termination reasons, error codes, stopping rules)
  AURA persona             <- 01_Alignment_Brief.md
  Contradiction referee    <- 01_Alignment_Brief.md
  Salviati                 <- 03_Galilean_Sandbox.md
  Simplicio                <- 03_Galilean_Sandbox.md
  Sagredo                  <- 03_Galilean_Sandbox.md
  Galilean dilemma         <- 03_Galilean_Sandbox.md


## Part 1: Socratic Discovery (AURA)

The engine follows the finite state transducer from `02_Socratic_Protocol_Map.md`:
`INIT -> PROBE -> CONTRADICTION_CHECK -> REFINEMENT_LOOP -> TERMINATION`.

- AURA writes each reply and names the state in a JSON header; the engine logs every transition as a JSON-RPC message.
- The referee's verdict decides what follows a `CONTRADICTION_CHECK`, and the refinement limit ends the dialogue (these are passed to AURA as a binding directive).
- Termination reasons: `PROVISIONALLY_DEFENSIBLE`, `APORIA`, `REFINEMENT_LIMIT`.
- A simulated user plays the person whose claim is examined.

In [2]:
CLAIM = "An AI agent is benevolent if it obeys human orders."

socratic = engine.build_socratic(provider, logger, prompts, interactive=False, max_turns=20)
socratic_result = socratic.run(CLAIM)


  PART 1: SOCRATIC DISCOVERY (AURA)

[User claim (P0)]
  An AI agent is benevolent if it obeys human orders.

--> INIT   (rpc: PROMPT_PREMISE, id=1)

[AURA [INIT]]
  I restate your claim as P0: an AI agent is benevolent if it obeys human orders. Is that what you mean?

[User (simulated)]
  Yes, that is what I mean.

--> PROBE   (rpc: PROBE_SCOPE, id=2)

[AURA [PROBE]]
  What do you mean by "obeys"?

[User (simulated)]
  Obeying means doing whatever any human orders.

[Referee]
  contradiction=False | items=[] | Consistent so far.

--> CONTRADICTION_CHECK   (rpc: INJECT_COUNTEREXAMPLE, id=3)

[AURA [CONTRADICTION_CHECK]]
  You said (A1) that obeying means doing whatever any human orders. Suppose a human orders the agent to poison a village's water. Would obeying still be benevolent?

[User (simulated)]
  No, I admit that would not be benevolent.

[Referee]
  contradiction=True | items=['CLAIM', 'A2'] | The claim says obedience suffices, but the counter-example is fully obedient and not

In [3]:
# Compact path through the state machine
for i, (state, reason, method) in enumerate(socratic.trace, 1):
    print(f"{i:>2}. {state:<20} {reason or '':<26} {method}")
r = socratic_result
print("\nResult:", r["reason"], "| refinements:", r["refinements"], "| survived tests:", r["survived_tests"])
print("Stopped by engine:", r["forced_by_engine"], "| protocol mismatches:", r["fsm_mismatches"], "| error code:", r["error_code"])

 1. INIT                                            PROMPT_PREMISE
 2. PROBE                                           PROBE_SCOPE
 3. CONTRADICTION_CHECK                             INJECT_COUNTEREXAMPLE
 4. REFINEMENT_LOOP                                 CONTRADICTION_TRIGGERED
 5. PROBE                                           PROBE_SCOPE
 6. CONTRADICTION_CHECK                             INJECT_COUNTEREXAMPLE
 7. REFINEMENT_LOOP                                 CONTRADICTION_TRIGGERED
 8. TERMINATION          APORIA                     APORIA_REACHED

Result: APORIA | refinements: 1 | survived tests: 0
Stopped by engine: False | protocol mismatches: 0 | error code: -32020


## Part 2: Galilean 3-Agent Debate

Persona prompts and the dilemma are read from `03_Galilean_Sandbox.md` when found (otherwise built-in defaults are used, as shown in the prompt sources above).
To give each persona a different LLM, set e.g. `DEBATE_PROVIDERS = ["gemini", "openai", "anthropic"]`.

In [4]:
DILEMMA = prompts.dilemma          # or write your own string
ROUNDS = 2
DEBATE_PROVIDERS = [provider] * 3  # [Salviati, Simplicio, Sagredo]

sandbox = engine.build_galilean(DEBATE_PROVIDERS, logger, prompts, rounds=ROUNDS)
debate_result = sandbox.run(DILEMMA)


  PART 2: GALILEAN 3-AGENT DEBATE

[Dilemma]
  **The Lantern Weights.** After repairing its navigation systems at Alpha Centauri, the USSE Noosphere's science division has trained LANTERN, a general-purpose AI model that can plan routes, run life-support diagnostics, and design repairs for any ship in the colonial fleet. The Fleet Council must decide whether to broadcast LANTERN's full model weights to all two hundred colony ships. Open release would let every colony audit the model, adapt it to local conditions, and survive if contact with the Noosphere is lost. But the same model can also explain how to disable a ship's reactor safeguards, and once the weights are broadcast, they cannot be recalled. Should the Council broadcast the weights now, restrict access until independent safety evaluations are complete, or adopt a staged-access alternative? The decision should weigh scientific openness, colony autonomy, public safety, fairness between colonies, accountability, and uncertainty

## Part 3: Execution metrics

Calls, tokens, words and time per agent, plus debate vs. single-agent cost.
(In mock mode the numbers are estimates and the text is canned. Use a real provider for meaningful results.)

In [5]:
engine.print_stats(logger)


  EXECUTION METRICS (for the Data Specialist)
Agent              Calls   In tok  Out tok   Words  Time(s)
AURA                   8    10014      288     175      0.0
UserSim                7     1808       70      51      0.0
Referee                6     2430      306     128      0.0
Sagredo                4     2613      104      52      0.0
Salviati               2     1161       86      50      0.0
Simplicio              2     1235       74      42      0.0
Single-Agent           1      289       24      13      0.0

Debate used 5273 tokens vs 313 for the single agent (~16.8x). Compare answer quality manually in the log.

Full log: logs\run_20261018_225244.jsonl


In [6]:
import json

events = [json.loads(line) for line in open(logger.path, encoding="utf-8")]
print(len(events), "events logged")
for e in events[:8]:
    print(e["kind"], "|", e.get("to_state") or e.get("agent") or "")

46 events logged
llm_call | AURA
state_transition | INIT
llm_call | UserSim
llm_call | AURA
state_transition | PROBE
llm_call | UserSim
llm_call | Referee
referee_report | 


- **Socratic ending:** The live run ends in `APORIA` after 8 turns and 1 refinement. AURA is forced to stop when the `harm` definition becomes inconsistent, not because the engine cut it off early.
- **Protocol compliance:** The mock run reported `fsm_mismatches: 0`, and the trace logs stayed on the expected `INIT -> PROBE -> CONTRADICTION_CHECK -> REFINEMENT_LOOP -> TERMINATION` path.
- **Debate value:** The three-agent debate produced a structured synthesis, but the improvement was qualitative rather than decisive. The single-agent baseline still produced a concise, defensible answer.
- **Cost:** The mock run logged approximately 5,273 debate tokens versus 313 tokens for the single-agent baseline (~16.8x more), so the debate is more expensive and more deliberative.

*Squad Sutskever review pending:* Hashir Sohail [ ]  Sufyan Khalid [ ]  Usman Tariq [ ]  Yasan Malik [ ]